<a href="https://colab.research.google.com/github/webomurga/dsai543-final/blob/main/Hyperparameter_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Initialization

In [ ]:
# @title
from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import subprocess
import torch
import gc
import tempfile
import cv2
import glob
import numpy as np
import pandas as pd
import itertools
import tifffile
from tqdm.notebook import tqdm
from IPython.display import display

print("Installing core dependencies...")
!pip install -q opencv-python-headless scikit-image pandas tqdm tifffile csbdeep cellpose gdown
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
!apt-get install -y openjdk-11-jdk-headless

BASE_DIR = "" # @param {"type":"string","placeholder":"Enter CellBinDB raw data directory's location here"}
DRIVE_RESULTS_DIR = "" # @param {"type":"string","placeholder":"Enter the desired results directory's location here."}

BASELINE_DIR = os.path.join(DRIVE_RESULTS_DIR, "experiment_baseline")

LOCAL_WEIGHTS_DIR = '/content/weights'
os.makedirs(LOCAL_WEIGHTS_DIR, exist_ok=True)

def setup_external_repos():
    if not os.path.exists('/content/hover_net'):
        print("Cloning HoVer-Net...")
        !git clone -q https://github.com/vqdang/hover_net.git
    if '/content/hover_net' not in sys.path:
        sys.path.insert(0, '/content/hover_net')

    if not os.path.exists('/content/MEDIAR'):
        print("Cloning MEDIAR...")
        !git clone -q https://github.com/Lee-Gihun/MEDIAR.git
    if '/content/MEDIAR' not in sys.path:
        sys.path.insert(0, '/content/MEDIAR')

setup_external_repos()

def download_weights():
    import gdown

    weights_to_download = {
        "sam_vit_h_4b8939.pth": "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth",
        "hovernet_weights.pth": "1FtoTDDnuZShZmQujjaFSLVJLD5sAh2_P",
    }

    for fname, source in weights_to_download.items():
        dest = os.path.join(LOCAL_WEIGHTS_DIR, fname)

        if os.path.exists(dest) and os.path.getsize(dest) > 1024 * 1024:
            print(f"{fname} exists and is valid.")
        else:
            print(f"Downloading {fname} to local storage...")
            if "http" in source:
                !wget -q -O "{dest}" "{source}"
            else:
                gdown.download(id=source, output=dest, quiet=False)

            if not os.path.exists(dest) or os.path.getsize(dest) < 1024 * 1024:
                print(f"ERROR: Failed to download {fname} correctly.")

download_weights()

SAM_CKPT = os.path.join(LOCAL_WEIGHTS_DIR, "sam_vit_h_4b8939.pth")
HOVER_CKPT = os.path.join(LOCAL_WEIGHTS_DIR, "hovernet_weights.pth")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Installing core dependencies...
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openjdk-11-jdk-headless is already the newest version (11.0.30+7-1ubuntu1~22.04).
0 upgraded, 0 newly installed, 0 to remove and 23 not upgraded.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  error: subpro

### Extensive Hyperparameter Search Space

For the second phase of the experiment (Failure Mode Recovery), an expanded grid search is utilized to identify the optimal balance between signal enhancement and noise introduction across four different segmentation architectures.

#### **H&E (CLAHE) Parameter Grid**
* **Clip Limit:** `[0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 7.5, 10.0]`
* **Tile Grid Size:** `[(4,4), (8,8), (12,12), (16,16), (24,24), (32,32)]`
* **Total Combinations:** 60

#### **mIF (Unsharp Masking) Parameter Grid**
* **Sigma ($\sigma$):** `[0.5, 0.8, 1.0, 1.2, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]`
* **Alpha ($\alpha$):** `[0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.5, 3.0]`
* **Total Combinations:** 90

---

### **Justification for Parameter Selection & Ranges**

The following table justifies the selection of these specific ranges based on the theoretical failure modes identified in the baseline study (e.g., low contrast in H&E and signal blurring in mIF).

| Technique | Parameter | Extensive Range | Theoretical Justification |
| :--- | :--- | :--- | :--- |
| **CLAHE** | `clipLimit` | **0.5 - 10.0** | Small values ($<1.5$) target standard contrast enhancement. The expansion to **10.0** specifically addresses "Worst" failure modes where necrosis or over-staining has flattened the dynamic range. This tests the limits of foundation models (SAM, MEDIAR) to distinguish between real edges and high-gain noise. |
| **CLAHE** | `tileGridSize` | **(4,4) - (32,32)** | Smaller grids (4x4) identify extremely local intensity variations in dense cell clusters. Larger grids (up to 32x32) ensure global consistency across the slide, preventing the "checkerboard" artifacting that often disrupts **HoVer-Net**'s horizontal and vertical distance branches. |
| **Unsharp** | `sigma` | **0.5 - 5.0** | The Gaussian standard deviation $\sigma$ defines the radius of the feature detection. Since mIF nuclei range significantly in size (lymphocytes vs. epithelial cells), a range up to **5.0** covers all biological scales while identifying the point where sharpening begins to create artificial halos. |
| **Unsharp** | `alpha` | **0.5 - 3.0** | $\alpha$ controls the gain of high-frequency components (edges). While standard values fall between 1.0 and 2.0, we include **3.0** to test the resilience of architectures against pixel-level background noise amplification, which is a common cause of False Positive detections in mIF. |

---

### **Mathematical Framework of Enhancement**

The recovery of failure modes through Unsharp Masking is governed by the enhancement of the high-pass components of the image:

$$I_{enhanced} = I_{original} + \alpha \cdot (I_{original} - G_{\sigma}(I_{original}))$$

Where $G_{\sigma}$ represents the Gaussian blur operator. The term $(I_{original} - G_{\sigma}(I_{original}))$ effectively isolates the edges (high frequencies). By performing a dense grid search over $\alpha$ and $\sigma$, we optimize the **Signal-to-Noise Ratio (SNR)** specifically to maximize the **Panoptic Quality (PQ)** metric, ensuring that the increase in cell detection (Recall) does not come at the cost of spatial accuracy or over-segmentation.

In [ ]:
def get_image_paths(modality_folder):
    base_path = os.path.join(BASE_DIR, modality_folder)
    img_paths = sorted(glob.glob(os.path.join(base_path, '**', '*-img.tif'), recursive=True))

    bin_paths, inst_paths = [], []
    for img_path in img_paths:
        dir_name = os.path.dirname(img_path)
        base_name = os.path.basename(img_path).replace('-img.tif', '')

        bin_paths.append(os.path.join(dir_name, f"{base_name}-mask.tif"))

        inst_p = os.path.join(dir_name, f"{base_name}-instancemask.tif")
        if not os.path.exists(inst_p):
            inst_p = os.path.join(dir_name, f"{base_name}-inst.npy")

        inst_paths.append(inst_p)

    return img_paths, bin_paths, inst_paths

def clean_id(filename):
    if not isinstance(filename, str): return ""
    base = os.path.basename(filename)
    for suffix in ['.tif', '.tiff', '.npy', '-img', '-mask', '-instancemask', '-inst']:
        base = base.replace(suffix, '')
    return base.strip()

def get_samples_from_baseline(model_name, modality, n_worst=4, n_median=3, n_best=3):
    if model_name == "HoVer-Net": model_name = "HoVerNet"
    csv_file = f"{model_name}_{modality}_results.csv"
    csv_path = os.path.join(BASELINE_DIR, csv_file)

    if not os.path.exists(csv_path):
        print(f"Baseline CSV not found: {csv_path}")
        return None

    try:
        df = pd.read_csv(csv_path)
        sort_col = 'Dice' if 'Dice' in df.columns else 'Panoptic Quality'
        df = df.sort_values(by=sort_col, ascending=True).reset_index(drop=True)

        worst = [clean_id(x) for x in df.head(n_worst)['Image']]
        mid_idx = len(df) // 2
        median = [clean_id(x) for x in df.iloc[mid_idx-n_median//2 : mid_idx+n_median//2+1]['Image']]
        best = [clean_id(x) for x in df.tail(n_best)['Image']]

        selected_ids = list(dict.fromkeys(worst + median + best))
        print(f"Success: Sampled {len(selected_ids)} IDs from {csv_file}")
        return selected_ids
    except Exception as e:
        print(f"Error sampling baseline: {e}")
        return None

def sync_data_triplets(img_paths, bin_paths, inst_paths, target_ids):
    bin_map = {clean_id(p): p for p in bin_paths}
    inst_map = {clean_id(p): p for p in inst_paths}

    s_i, s_b, s_inst = [], [], []

    for i_p in img_paths:
        current_id = clean_id(i_p)

        if target_ids is None or current_id in target_ids:
            if current_id in bin_map and current_id in inst_map:
                s_i.append(i_p)
                s_b.append(bin_map[current_id])
                s_inst.append(inst_map[current_id])

    if not s_i and target_ids is not None:
        print(f"DEBUG: No matches found. Sample Data ID: '{clean_id(img_paths[0])}' | Sample Target ID: '{target_ids[0]}'")

    return s_i, s_b, s_inst

def tune_hyperparameters(img_paths, bin_paths, inst_paths, modality, model_wrapper, num_val_images):
    evaluator = Evaluator()

    if modality == "HE":
        grid = {
            # clipLimit: Granular steps from subtle to aggressive
            'clip': [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 7.5, 10.0],
            # tileGridSize: From fine-grained local to broader regional contrast
            'grid': [(4,4), (8,8), (12,12), (16,16), (24,24), (32,32)]
        }
    else:
        # mIF: Unsharp Masking
        grid = {
            # sigma: Radius of feature detection (Standard Nuclei are ~1-3.0)
            'sigma': [0.5, 0.8, 1.0, 1.2, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0],
            # alpha: Blending weight for the sharpened edge
            'alpha': [0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.5, 3.0]
        }
    keys, combinations = grid.keys(), list(itertools.product(*grid.values()))

    all_tuning_results = []

    for combo in tqdm(combinations, desc=f"Grid Search ({modality})", leave=False):
        params = dict(zip(keys, combo))

        if modality == "HE":
            enhancer = HyperTuningEnhancer(clahe_clip=params['clip'], clahe_grid=params['grid'])
        else:
            enhancer = HyperTuningEnhancer(unsharp_sigma=params['sigma'], unsharp_alpha=params['alpha'])

        dices, pqs = [], []

        for i in range(min(num_val_images, len(img_paths))):
            raw = cv2.cvtColor(cv2.imread(img_paths[i]), cv2.COLOR_BGR2RGB)
            gt_bin = cv2.imread(bin_paths[i], cv2.IMREAD_UNCHANGED)

            if inst_paths[i].endswith('.npy'):
                gt_inst = np.load(inst_paths[i])
            else:
                gt_inst = cv2.imread(inst_paths[i], cv2.IMREAD_UNCHANGED)

            processed = enhancer.apply(raw, modality)
            pred_mask = model_wrapper.predict(processed)

            metrics = evaluator.calculate_metrics(pred_mask, gt_bin, gt_inst)
            dices.append(metrics['Dice'])
            pqs.append(metrics['PQ'])

        result_entry = {**params, 'Mean_Dice': np.mean(dices), 'Mean_PQ': np.mean(pqs)}
        all_tuning_results.append(result_entry)

    return pd.DataFrame(all_tuning_results).sort_values(by='Mean_PQ', ascending=False)

In [ ]:
class HyperTuningEnhancer:
    def __init__(self, clahe_clip=3.0, clahe_grid=(8,8), unsharp_sigma=2.0, unsharp_alpha=1.5):
        self.clahe = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=clahe_grid)
        self.u_sigma = unsharp_sigma
        self.u_alpha = unsharp_alpha

    def apply(self, img, modality):
        if modality == "HE":
            lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
            l, a, b = cv2.split(lab)
            l = self.clahe.apply(l)
            return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)
        else:
            if len(img.shape) == 3:
                img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
            gauss = cv2.GaussianBlur(img, (0, 0), self.u_sigma)
            return cv2.addWeighted(img, self.u_alpha, gauss, (1 - self.u_alpha), 0)

class Evaluator:
    def __init__(self):
        pass

    @staticmethod
    def calculate_panoptic_quality(pred_inst, true_inst, iou_threshold=0.5):
        true_ids = np.unique(true_inst)[1:]
        pred_ids = np.unique(pred_inst)[1:]

        tp_iou_sum, tp = 0, 0
        fp, fn = len(pred_ids), len(true_ids)

        for t_id in true_ids:
            t_mask = (true_inst == t_id)
            best_iou = 0
            for p_id in pred_ids:
                p_mask = (pred_inst == p_id)
                intersection = np.logical_and(t_mask, p_mask).sum()
                union = np.logical_or(t_mask, p_mask).sum()
                iou = intersection / union if union > 0 else 0
                if iou > best_iou:
                    best_iou = iou

            if best_iou >= iou_threshold:
                tp += 1
                fp -= 1
                fn -= 1
                tp_iou_sum += best_iou

        pq = tp_iou_sum / (tp + 0.5 * fp + 0.5 * fn) if (tp + 0.5 * fp + 0.5 * fn) > 0 else 0
        return pq

    def calculate_metrics(self, pred_mask, gt_bin, gt_inst):
        pred_bin = (pred_mask > 0).astype(np.uint8)
        gt_bin = (gt_bin > 0).astype(np.uint8)

        intersection = np.logical_and(pred_bin, gt_bin).sum()
        dice = (2. * intersection) / (pred_bin.sum() + gt_bin.sum() + 1e-7)
        tp = intersection
        fp = pred_bin.sum() - tp
        fn = gt_bin.sum() - tp
        precision = tp / (tp + fp + 1e-7)
        recall = tp / (tp + fn + 1e-7)

        pq = self.calculate_panoptic_quality(pred_mask, gt_inst)

        return {
            "Dice": dice,
            "Precision": precision,
            "Recall": recall,
            "PQ": pq
        }

## Cellpose, HoVerNet & SAM code

In [ ]:
class CellposeWrapper:
    def __init__(self):
        from cellpose import models
        self.model = models.CellposeModel(gpu=True)
    def predict(self, img):
        masks, _, _ = self.model.eval(img, diameter=None, channels=[0,0])
        return masks

class HoVerNetWrapper:
    def __init__(self, checkpoint=HOVER_CKPT):
        from models.hovernet.net_desc import HoVerNet
        self.model = HoVerNet(nr_types=None)
        state_dict = torch.load(checkpoint, map_location='cuda')['desc']
        self.model.load_state_dict(state_dict, strict=False)
        self.model.to("cuda").eval()

    def predict(self, img):
        h, w = img.shape[:2]

        if len(img.shape) == 2:
            img_input = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        else:
            img_input = img

        img_p = cv2.resize(img_input, (270, 270)) / 255.0

        inp = torch.from_numpy(img_p).permute(2,0,1).float().unsqueeze(0).to("cuda")

        with torch.no_grad():
            out = self.model(inp)['np'].squeeze().cpu().numpy()

        mask = (out[..., 1] > 0.5).astype(np.float32)
        resized_mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)

        return (resized_mask > 0.5).astype(np.int32)

class SAMWrapper:
    def __init__(self, checkpoint=SAM_CKPT):
        from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
        sam = sam_model_registry["vit_h"](checkpoint=checkpoint).to("cuda")
        self.gen = SamAutomaticMaskGenerator(sam)
    def predict(self, img):
        if len(img.shape) == 2: img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        res = self.gen.generate(img)
        mask = np.zeros(img.shape[:2], dtype=np.int32)
        for i, ann in enumerate(sorted(res, key=lambda x: x['area'], reverse=True)):
            mask[ann['segmentation']] = i + 1
        return mask

In [ ]:
model_classes = {
    "Cellpose": CellposeWrapper,
    "SAM": SAMWrapper,
    "HoVer-Net": HoVerNetWrapper,
}

for name in model_classes.keys():
    print(f"\n{'='*40}\nSTARTING TUNING: {name}\n{'='*40}")

    try:
        model_instance = model_classes[name]()
        print(f"{name} loaded into VRAM.")
    except Exception as e:
        print(f"Failed to initialize {name}: {e}")
        continue

    for mod in ["HE", "mIF"]:
        img_l, bin_l, inst_l = get_image_paths(mod)
        target_ids = get_samples_from_baseline(name, mod)

        i_rep, b_rep, inst_rep = sync_data_triplets(img_l, bin_l, inst_l, target_ids)

        if not i_rep:
            i_rep, b_rep, inst_rep = i_rep[:10], b_rep[:10], inst_rep[:10]

        if i_rep:
            print(f"Tuning {name} on {mod}...")
            tuning_df = tune_hyperparameters(i_rep, b_rep, inst_rep, mod, model_instance, len(i_rep))

            print(f"\nTOP 3 CONFIGURATIONS FOR {name} ({mod}):")

            top_3 = tuning_df.head(3).copy()

            best_params = top_3.iloc[0]
            print(f"BEST: {best_params.to_dict()}")

            display(top_3)

            save_path = os.path.join(DRIVE_RESULTS_DIR, f"{mod}_{name}_Tuning_Results.csv")
            tuning_df.to_csv(save_path, index=False)

    print(f"Cleaning up {name} from VRAM...")
    del model_instance
    gc.collect()
    torch.cuda.empty_cache()

    import time
    time.sleep(2)


STARTING TUNING: SAM
SAM loaded into VRAM.
Success: Sampled 7 IDs from SAM_mIF_results.csv
Tuning SAM on mIF...


Grid Search (mIF):   0%|          | 0/90 [00:00<?, ?it/s]


TOP 3 CONFIGURATIONS FOR SAM (mIF):
BEST: {'sigma': 1.2, 'alpha': 2.0, 'Mean_Dice': 0.8923946480997118, 'Mean_PQ': 0.8425422381508357}


,sigma,alpha,Mean_Dice,Mean_PQ
33,1.2,2.0,0.892395,0.842542
26,1.0,3.0,0.863922,0.819657
16,0.8,2.5,0.861970,0.818081


Cleaning up SAM from VRAM...


## DeepCell code

In [ ]:
%%bash
add-apt-repository ppa:deadsnakes/ppa -y > /dev/null 2>&1
apt-get update > /dev/null 2>&1
apt-get install python3.10 python3.10-distutils -y > /dev/null 2>&1
curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10 > /dev/null 2>&1

wget -q https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb
dpkg -i cuda-keyring_1.1-1_all.deb > /dev/null 2>&1
apt-get update > /dev/null 2>&1
apt-get install -y cuda-toolkit-11-8 libcudnn8 > /dev/null 2>&1

python3.10 -m pip install -q deepcell opencv-python-headless scikit-image pandas > /dev/null 2>&1

In [ ]:
%%writefile run_deepcell.py
import os
import sys

os.environ['LD_LIBRARY_PATH'] = f"/usr/local/cuda-11.8/lib64:{os.environ.get('LD_LIBRARY_PATH', '')}"
os.environ["DEEPCELL_ACCESS_TOKEN"] = "K8XjkycL.dxfcOwsm7FFMR6JYTApMkYe5Eo27UDc7"

import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
import gc
from deepcell.applications import Mesmer
from skimage.color import separate_stains, hdx_from_rgb
from skimage.filters import unsharp_mask
from tqdm.notebook import tqdm

def get_completed_images(csv_path):
    if os.path.exists(csv_path):
        try:
            df = pd.read_csv(csv_path)
            if 'Image' in df.columns:
                return set(df['Image'].unique())
        except pd.errors.EmptyDataError:
            return set()
    return set()


class DeepCellWrapper:
    def __init__(self):
        print("Loading DeepCell Mesmer weights...")
        self.model = Mesmer()

    def predict(self, image):
        if len(image.shape) == 3 and image.shape[-1] == 3:
            image = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

        blank_channel = np.zeros_like(image)
        image_2ch = np.stack((image, blank_channel), axis=-1)
        img_expanded = np.expand_dims(image_2ch, axis=0)
        prediction = self.model.predict(img_expanded, image_mpp=0.5, compartment='nuclear')

        return prediction[0, ..., 0]

def get_cellbindb_paths(base_dir):
    image_paths, binary_mask_paths, instance_mask_paths = [], [], []
    subdirs = [os.path.join(base_dir, d) for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
    for subdir in subdirs:
        folder_name = os.path.basename(subdir)
        img_path, bin_mask_path, inst_mask_path = os.path.join(subdir, f"{folder_name}-img.tif"), os.path.join(subdir, f"{folder_name}-mask.tif"), os.path.join(subdir, f"{folder_name}-instancemask.tif")
        if not os.path.exists(img_path): img_path += "f"; bin_mask_path += "f"; inst_mask_path += "f"
        if os.path.exists(img_path) and os.path.exists(bin_mask_path) and os.path.exists(inst_mask_path):
            image_paths.append(img_path); binary_mask_paths.append(bin_mask_path); instance_mask_paths.append(inst_mask_path)
    return image_paths, binary_mask_paths, instance_mask_paths

import itertools

class HyperTuningEnhancer:
    def __init__(self, clahe_clip=3.0, clahe_grid=(8,8), unsharp_sigma=2.0, unsharp_alpha=1.5):
        self.clahe = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=clahe_grid)
        self.u_sigma = unsharp_sigma
        self.u_alpha = unsharp_alpha

    def apply(self, img, modality):
        if modality == "HE":
            lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
            l, a, b = cv2.split(lab)
            l = self.clahe.apply(l)
            return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)
        else:
            if len(img.shape) == 3:
                img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
            gauss = cv2.GaussianBlur(img, (0, 0), self.u_sigma)
            return cv2.addWeighted(img, self.u_alpha, gauss, (1 - self.u_alpha), 0)

class Evaluator:
    def __init__(self):
        pass

    def calculate_metrics(self, pred_mask, gt_bin, gt_inst):
        pred_bin = (pred_mask > 0).astype(np.uint8)
        gt_bin = (gt_bin > 0).astype(np.uint8)

        intersection = np.logical_and(pred_bin, gt_bin).sum()
        dice = (2. * intersection) / (pred_bin.sum() + gt_bin.sum() + 1e-7)

        tp = intersection
        fp = pred_bin.sum() - tp
        fn = gt_bin.sum() - tp

        precision = tp / (tp + fp + 1e-7)
        recall = tp / (tp + fn + 1e-7)

        pq = dice * precision

        return {
            "Dice": dice,
            "Precision": precision,
            "Recall": recall,
            "PQ": pq
        }

def run_modality(i_paths, b_paths, inst_paths, modality, out_csv):
    if not i_paths: return
    evaluator = Evaluator()
    model = DeepCellWrapper()

    if modality == "HE":
        grid = {
            'clip': [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 7.5, 10.0],
            'grid': [(4,4), (8,8), (12,12), (16,16), (24,24), (32,32)]
        }
    else:
        grid = {
            'sigma': [0.5, 0.8, 1.0, 1.2, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0],
            'alpha': [0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.5, 3.0]
        }

    keys, combinations = grid.keys(), list(itertools.product(*grid.values()))

    completed_combos = set()
    if os.path.exists(out_csv):
        try:
            existing_df = pd.read_csv(out_csv)
            if all(k in existing_df.columns for k in keys):
                completed_combos = set(tuple(row) for row in existing_df[keys].itertuples(index=False, name=None))
                print(f"[{modality}] Found existing CSV. Skipping {len(completed_combos)} completed combinations.")
        except Exception as e:
            print(f"[{modality}] Warning: Could not read existing CSV to resume ({e}). Starting fresh.")

    combinations = [c for c in all_combinations if c not in completed_combos]

    if not combinations:
        print(f"[{modality}] All combinations already completed! Skipping.")
        return

    num_val_images = min(10, len(i_paths))

    all_tuning_results = []

    for combo in tqdm(combinations, desc=f"Grid Search ({modality})"):
        params = dict(zip(keys, combo))

        if modality == "HE":
            enhancer = HyperTuningEnhancer(clahe_clip=params['clip'], clahe_grid=params['grid'])
        else:
            enhancer = HyperTuningEnhancer(unsharp_sigma=params['sigma'], unsharp_alpha=params['alpha'])

        dices, pqs = [], []

        for i in range(num_val_images):
            raw_img = cv2.imread(i_paths[i])
            if raw_img is None: continue
            raw_img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)

            gt_bin_mask = cv2.imread(b_paths[i], cv2.IMREAD_UNCHANGED)
            gt_inst_mask = cv2.imread(inst_paths[i], cv2.IMREAD_UNCHANGED)

            processed_img = enhancer.apply(raw_img, modality)

            pred_mask = model.predict(processed_img)

            metrics = evaluator.calculate_metrics(pred_mask, gt_bin_mask, gt_inst_mask)
            dices.append(metrics['Dice'])
            pqs.append(metrics['PQ'])

        result_entry = {**params, 'Mean_Dice': np.mean(dices), 'Mean_PQ': np.mean(pqs)}
        all_tuning_results.append(result_entry)

        df_batch = pd.DataFrame([result_entry])
        write_header = not os.path.exists(out_csv)
        df_batch.to_csv(out_csv, mode='a', header=write_header, index=False)

        gc.collect()
        tf.keras.backend.clear_session()

def clean_id(filename):
    if not isinstance(filename, str): return ""
    base = os.path.basename(filename)
    for suffix in ['.tif', '.tiff', '.npy', '-img', '-mask', '-instancemask', '-inst']:
        base = base.replace(suffix, '')
    return base.strip()

def get_samples_from_baseline(baseline_dir, model_name, modality, n_worst=4, n_median=3, n_best=3):
    csv_file = f"{model_name}_{modality}_results.csv"
    csv_path = os.path.join(baseline_dir, csv_file)

    if not os.path.exists(csv_path):
        print(f"Baseline CSV not found: {csv_path}. Falling back to first 10 images.")
        return None

    try:
        df = pd.read_csv(csv_path)
        sort_col = 'Dice' if 'Dice' in df.columns else 'Panoptic Quality'
        df = df.sort_values(by=sort_col, ascending=True).reset_index(drop=True)

        worst = [clean_id(x) for x in df.head(n_worst)['Image']]
        mid_idx = len(df) // 2
        median = [clean_id(x) for x in df.iloc[mid_idx-n_median//2 : mid_idx+n_median//2+1]['Image']]
        best = [clean_id(x) for x in df.tail(n_best)['Image']]

        selected_ids = list(dict.fromkeys(worst + median + best))
        print(f"Success: Sampled {len(selected_ids)} IDs from {csv_file}")
        return selected_ids
    except Exception as e:
        print(f"Error sampling baseline: {e}")
        return None

def sync_data_triplets(img_paths, bin_paths, inst_paths, target_ids):
    bin_map = {clean_id(p): p for p in bin_paths}
    inst_map = {clean_id(p): p for p in inst_paths}

    s_i, s_b, s_inst = [], [], []

    for i_p in img_paths:
        current_id = clean_id(i_p)
        if target_ids is None or current_id in target_ids:
            if current_id in bin_map and current_id in inst_map:
                s_i.append(i_p)
                s_b.append(bin_map[current_id])
                s_inst.append(inst_map[current_id])

    return s_i, s_b, s_inst

def main():
    if len(sys.argv) < 3:
        raise ValueError(
            "Error: BASE_DIR and DRIVE_RESULTS_DIR must be supplied as command-line arguments.\n"
            "Usage: python run_deepcell.py /path/to/BASE_DIR /path/to/DRIVE_RESULTS_DIR"
        )

    BASE_DIR = sys.argv[1]
    DRIVE_RESULTS_DIR = sys.argv[2]
    BASELINE_DIR = os.path.join(DRIVE_RESULTS_DIR, "experiment_baseline")

    HE_BASE_DIR = os.path.join(BASE_DIR, "HE")
    MIF_BASE_DIR = os.path.join(BASE_DIR, "mIF")

    print("\nPreparing HE Data...")
    i_paths, b_paths, inst_paths = get_cellbindb_paths(HE_BASE_DIR)
    target_ids_he = get_samples_from_baseline(BASELINE_DIR, "DeepCell", "HE")
    s_i_he, s_b_he, s_inst_he = sync_data_triplets(i_paths, b_paths, inst_paths, target_ids_he)

    if not s_i_he:
        s_i_he, s_b_he, s_inst_he = i_paths[:10], b_paths[:10], inst_paths[:10]

    run_modality(s_i_he, s_b_he, s_inst_he, "HE", os.path.join(DRIVE_RESULTS_DIR, "HE_DeepCell_Tuning_Results.csv"))

    print("\nPreparing mIF Data...")
    i_paths, b_paths, inst_paths = get_cellbindb_paths(MIF_BASE_DIR)
    target_ids_mif = get_samples_from_baseline(BASELINE_DIR, "DeepCell", "mIF")
    s_i_mif, s_b_mif, s_inst_mif = sync_data_triplets(i_paths, b_paths, inst_paths, target_ids_mif)

    if not s_i_mif:
        s_i_mif, s_b_mif, s_inst_mif = i_paths[:10], b_paths[:10], inst_paths[:10]

    run_modality(s_i_mif, s_b_mif, s_inst_mif, "mIF", os.path.join(DRIVE_RESULTS_DIR, "mIF_DeepCell_Tuning_Results.csv"))

if __name__ == "__main__":
    main()

Overwriting run_deepcell.py


In [ ]:
!python3.10 run_deepcell.py "{BASE_DIR}" "{DRIVE_RESULTS_DIR}"


Preparing HE Data...
Success: Sampled 10 IDs from DeepCell_HE_results.csv
Loading DeepCell Mesmer weights...
INFO:root:Checking for cached data
INFO:root:Checking MultiplexSegmentation-9.tar.gz against provided file_hash...
INFO:root:MultiplexSegmentation-9.tar.gz with hash a1dfbce2594f927b9112f23a0a1739e0 already available.
INFO:root:Extracting /root/.deepcell/models/MultiplexSegmentation-9.tar.gz
INFO:root:Successfully extracted /root/.deepcell/models/MultiplexSegmentation-9.tar.gz into /root/.deepcell/models
2026-03-29 19:03:15.942599: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/lib/python3.10/dist-packages/cv2/../../lib64:/usr/local/cuda-11.8/lib64:/usr/local/lib/python3.12/dist-packages/cv2/../../lib64:
2026-03-29 19:03:15.944786: W tensorflow/stream_executor/cuda/cuda_driver.cc:269] failed call to cuInit: